# Example 6.1 (the Beta distributions) in CT+iDCI paper

## Setup libraries and helper functions

In [ ]:
# The libraries we will use
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde as GKDE
from scipy.stats import norm, beta, uniform, multivariate_normal
import scipy.io as sio
from scipy.stats import entropy
from tqdm import tqdm
import matplotlib.colors as mcolors

In [ ]:
from ipywidgets import interact, IntSlider
import ipywidgets as widgets

### Define iDCI function

In [ ]:
def iterative_DCI(Q_pred_vals, observed_densities, num_epochs=1, r_init=1, 
                  pred_tol=1e-12, ratio_tol=0.1, KL_tol=1e-7, KL_update_rel_tol = 1e-2,
                  verbose=False, QoI_spaces=[]):
    '''
    Assume Q_pred_vals is a (num_samples x QoI_num) array of QoI vals 
    associated with initial parameter samples and observed_densities 
    is a list of observed QoI densities for each QoI map (coming from a
    data generating distribution)
    '''
    num_samples, num_QoI = np.shape(Q_pred_vals)

    if len(QoI_spaces) == 0:
        QoI_spaces = list(np.arange(num_QoI))
    
    true_obs_probs = []
    for d in range(len(QoI_spaces)):
        true_obs_probs.append(observed_densities[d](Q_pred_vals[:,QoI_spaces[d]].T))
    
    with tqdm(total=100) as pbar:    
        
        if np.size(r_init)==1:  # No initial weighting (possibly from prior iterations)
            rs = np.ones((num_samples, len(QoI_spaces), num_epochs))
            r_current = np.ones(num_samples)
        else:
            r_current = np.copy(r_init)
            rs = np.ones((num_samples, len(QoI_spaces), num_epochs))
            
        kl_from_observed_marginal = np.zeros((len(QoI_spaces), num_epochs))
        
        for k in range(num_epochs):
            
            if verbose is True:
                print('Starting epoch #', k)
            
            for d in range(len(QoI_spaces)):
                r_current /= np.mean(r_current)
                if verbose is True: 
                    print('Iteration #', d)
                    print('Some current r-values:', r_current[0:3])
                pred_dens = GKDE(Q_pred_vals[:,QoI_spaces[d]].T,
                                 weights=r_current)
                rs_update = np.where(pred_dens(Q_pred_vals[:,QoI_spaces[d]].T)>pred_tol,
                                        np.divide(observed_densities[d](Q_pred_vals[:,QoI_spaces[d]].T),
                                        pred_dens(Q_pred_vals[:,QoI_spaces[d]].T)), 0)
                r_current *= rs_update
                rs[:,d,k] = r_current
                if np.abs(np.mean(r_current)-1.0)>ratio_tol:
                    print('Outside of ratio tolerance at epoch #', k+1, ' iteration #', d+1)
                    return rs[:,:d+1,:k+1], k+1, d+1, kl_from_observed_marginal[:,k]
                pbar.update(100/(len(QoI_spaces)*num_epochs))
            
            for d in range(len(QoI_spaces)):
                pred_dens = GKDE(Q_pred_vals[:, QoI_spaces[d]].T, 
                                 weights=r_current)
                current_probs = pred_dens(Q_pred_vals[:,QoI_spaces[d]].T)
                kl_from_observed_marginal[d, k] = entropy(true_obs_probs[d], current_probs) / num_samples
            
            if verbose is True:
                print('Epoch #', k)
                print('KL divergences from observed marginals', kl_from_observed_marginal[:,k])
            
            if all( kl < KL_tol for kl in kl_from_observed_marginal[:,k]):
                print('KL divergences from observed marginals all within tolerance at epoch #', k+1)
                return rs[:,:,:k+1], k+1, d+1, kl_from_observed_marginal
            
            if k>0:
                kl_relative_update = np.abs(kl_from_observed_marginal[:,k-1] - kl_from_observed_marginal[:,k]) /\
                                    kl_from_observed_marginal[:,k-1]
                if all(kl_rel < KL_update_rel_tol for kl_rel in kl_relative_update):
                    print('KL divergences from observed marginals are not sufficiently updated at epoch #', k+1)
                    return rs[:,:,:k+1], k+1, d+1, kl_from_observed_marginal
                    
    return rs, k+1, d+1, kl_from_observed_marginal

### Define functions for the approximate copula transformation

In [ ]:
def weighted_correlation(x, y, w):
    """Calculate the weighted Pearson correlation coefficient between two vectors."""
    x, y, w = np.array(x), np.array(y), np.array(w)
    
    # 1. Calculate weighted means
    mean_x = np.sum(x * w) / np.sum(w)
    mean_y = np.sum(y * w) / np.sum(w)
    
    # 2. Calculate weighted variances and covariance
    cov_xy = np.sum(w * (x - mean_x) * (y - mean_y)) / np.sum(w)
    cov_xx = np.sum(w * (x - mean_x) ** 2) / np.sum(w)
    cov_yy = np.sum(w * (y - mean_y) ** 2) / np.sum(w)
    
    # 3. Calculate weighted correlation
    return cov_xy / np.sqrt(cov_xx * cov_yy)

In [ ]:
def copula_transformation(rs, Q_pred_vals):
    
    rho_hat_pred = weighted_correlation(Q_pred_vals[:,0], Q_pred_vals[:,1], rs)
    rho_hat_obs = np.corrcoef(Q_obs_vals[:,0], Q_obs_vals[:,1])[0,1]

    # Use CDFs of the observed distributions to map to the shared
    # copula coordinate system
    def Fx_interp(x):
        if isinstance(x, float):
            return observed_densities[0].integrate_box_1d(-np.inf, x)
        else:
            u1 = np.zeros(len(x))
            for i in range(len(x)):
                u1[i] = observed_densities[0].integrate_box_1d(-np.inf, x[i])
            return u1
    
    def Fy_interp(x):
        if isinstance(x, float):
            return observed_densities[1].integrate_box_1d(-np.inf, x)
        else:
            u2 = np.zeros(len(x))
            for i in range(len(x)):
                u2[i] = observed_densities[1].integrate_box_1d(-np.inf, x[i])
            return u2

    # Define Gaussian copula function
    def copula_Gaussian(rho, data_1, data_2):
        return multivariate_normal.pdf(np.dstack((data_1, data_2)),
                                       cov = [[1, rho], [rho, 1]]                                        )
    
    # Transform predictions to pseudo-obs space 
    # this requires the weighted eCDFs in general
    u = Fx_interp(Q_pred_vals[:,0])
    v = Fy_interp(Q_pred_vals[:,1])
    
    # Now ensure we avoid the boundary of the unit hyperbox
    eps = 1e-15
    u = np.clip(u, eps, 1-eps)
    v = np.clip(v, eps, 1-eps)
    
    # Transform to Gaussian scores
    z = norm.ppf(u)
    q = norm.ppf(v)
    
    # Evaluate the predicted and observed copulas on their shared coordinate system
    copula_obs = copula_Gaussian(rho_hat_obs, z, q)
    copula_pred = copula_Gaussian(rho_hat_pred, z, q)
    
    copula_r = np.divide(copula_obs, copula_pred)

    copula_transformed_r = copula_r * rs

    return copula_transformed_r, copula_r

### Create 2d plotting functions

`plot_epoch_2d_params` plots the updates and QoI densities as a function of epoch number

`plot_2d_params_compare` allows us to compare different parameter distributions

In [ ]:
def plot_epoch_2d_params(fignum, epoch_num, observed_densities, rs, 
                         fix_cbar_ranges=False, cmap='hot_r', savefig=False,
                         filename='file.png'):

    # Construct updated densities for particular iterations of epoch
    update_dens_iter_1 = GKDE(params_init.T,
                              weights = rs[:,0,epoch_num-1])
    update_dens_iter_2 = GKDE(params_init.T,
                              weights = rs[:,1,epoch_num-1])
    if num_QoI==3:
        update_dens_iter_3 = GKDE(params_init.T, 
                              weights = rs[:,2,epoch_num-1])
    
    kde_truth = GKDE(params_dg.T)

    # Plot settings defined as dict
    plot_dict = {'marker' : 'o', 
                 'cmap' : cmap, 
                 's': 10}
    if fix_cbar_ranges is True:
        true_vals = kde_truth(params_init.T)
        true_max = np.max(true_vals)
        true_min = np.min(true_vals)
        plot_dict['vmin'] = true_min
        plot_dict['vmax'] = true_max

    num_points = 100
    x = np.linspace(0, 1, num_points)
    y = np.linspace(0, 1, num_points)
    
    X, Y = np.meshgrid(x, y)
    plotting_params = np.vstack([X.ravel(), Y.ravel()]).T

    # Create plots
    fig = plt.figure(num=fignum, figsize=(9, 6), tight_layout=True)
    fig.clf()
    gs = fig.add_gridspec(2,6)
    
    ax1 = fig.add_subplot(gs[0,0:2])
    scatter1 = ax1.scatter(plotting_params[:,0], plotting_params[:,1],
                c = update_dens_iter_1(plotting_params.T),
                **plot_dict)
    ax1.set_aspect('equal', adjustable='box')
    title_str = '1st iter. of epoch ' + str(epoch_num)
    ax1.set_title(title_str)
    ax1.set_xlabel(r'$\lambda_1$')
    ax1.set_ylabel(r'$\lambda_2$')
    plt.colorbar(scatter1, ax=ax1, fraction=0.046, pad=0.04)
    
    
    ax2 = fig.add_subplot(gs[0,2:4])
    scatter2 = ax2.scatter(plotting_params[:,0], plotting_params[:,1],
                c = update_dens_iter_2(plotting_params.T), 
                **plot_dict)
    ax2.set_aspect('equal', adjustable='box')
    title_str = '2nd iter. of epoch ' + str(epoch_num)
    ax2.set_title(title_str)
    ax2.set_xlabel(r'$\lambda_1$')
    ax2.set_ylabel(r'$\lambda_1$')
    plt.colorbar(scatter2, ax=ax2, fraction=0.046, pad=0.04)
    
    ax4 = fig.add_subplot(gs[0,4:])
    scatter4 = ax4.scatter(plotting_params[:,0], plotting_params[:,1],
                c = kde_truth(plotting_params.T),
                **plot_dict)
    ax4.set_aspect('equal', adjustable='box')
    ax4.set_title('KDE of DG')
    ax4.set_xlabel(r'$\lambda_1$')
    ax4.set_ylabel(r'$\lambda_1$')
    plt.colorbar(scatter4, ax=ax4, fraction=0.046, pad=0.04)

    ax5 = fig.add_subplot(gs[1,0:3])
    pred_dens = GKDE(Q_pred_vals[:,0],weights=rs[:,1,epoch_num-1])
    min_Q = np.min(Q_pred_vals[:,0])
    max_Q = np.max(Q_pred_vals[:,0])
    plt_Qs = np.linspace(min_Q-0.05*(max_Q-min_Q), max_Q+0.05*(max_Q-min_Q), 101)
    ax5.plot(plt_Qs, pred_dens(plt_Qs), linestyle=':', linewidth=2, label='PF of Update')
    ax5.plot(plt_Qs, observed_densities[0](plt_Qs), label='Observed')
    ax5.set_xlabel(r'$Q_1$')
    ax5.set_title('$Q_1$ densities after epoch ' + str(epoch_num))
    ax5.legend()
    
    ax6 = fig.add_subplot(gs[1,3:])
    pred_dens = GKDE(Q_pred_vals[:,1],weights=rs[:,1,epoch_num-1])
    min_Q = np.min(Q_pred_vals[:,1])
    max_Q = np.max(Q_pred_vals[:,1])
    plt_Qs = np.linspace(min_Q-0.05*(max_Q-min_Q), max_Q+0.05*(max_Q-min_Q), 101)
    ax6.plot(plt_Qs, pred_dens(plt_Qs), linestyle=':', linewidth=2, label='PF of Update')
    ax6.plot(plt_Qs, observed_densities[1](plt_Qs), label='Observed')
    ax6.set_xlabel(r'$Q_2$')
    ax6.set_title('$Q_2$ densities after epoch ' + str(epoch_num))
    ax6.legend()

    plt.tight_layout(pad=0.25)
    if savefig is True:
        plt.savefig(filename, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

In [ ]:
def plot_2d_params_compare(fignum, rs1, rs2, which_params=[0, 1], cmap='viridis_r',
                           ax1_title = 'iDCI Solution', 
                           ax2_title = 'CT+iDCI Solution',
                           exact_dg = False,
                           fix_cbar_ranges = True,
                           savefig=False,
                           filename='file.png'):

    # Construct updated densities for particular iterations of epoch
    update_dens_1 = GKDE(params_init[:, which_params].T,
                              weights = rs1)
    update_dens_2 = GKDE(params_init[:, which_params].T,
                              weights = rs2)

    if exact_dg is False:
        dg_dens = GKDE(params_dg[:, which_params].T)
    else:
        dg_dens = Beta_dg_pdf
        
    num_points = 100
    x = np.linspace(0, 1, num_points)
    y = np.linspace(0, 1, num_points)
    
    X, Y = np.meshgrid(x, y)
    plotting_params = np.vstack([X.ravel(), Y.ravel()]).T
    
    Q_vals_1 = Q_maps[0](plotting_params)
    Q_vals_2 = Q_maps[1](plotting_params)
    
    # Plot settings defined as dict
    base_cmap = plt.get_cmap(cmap)
    shared_levels=20
    color_matrix = base_cmap(np.linspace(0, 1, shared_levels-1))
    
    # Force the first 4 rows (bottom 4 levels) to be white
    color_matrix[0, :] = [1.0, 1.0, 1.0, 1.0] # [R, G, B, Alpha]
    custom_cmap = mcolors.ListedColormap(color_matrix)
    # Plot settings defined as dict
    plot_dict = {'cmap' : custom_cmap}
    
    if fix_cbar_ranges is True:
        true_vals = dg_dens(params_init[:, which_params].T)
        up_vals_1 = update_dens_1(params_init[:, which_params].T)
        up_vals_2 = update_dens_2(params_init[:, which_params].T)
        true_max = np.max([true_vals.max(), up_vals_1.max(), up_vals_2.max()])
        true_min = np.min(true_vals)
        plot_dict['vmin'] = true_min
        plot_dict['vmax'] = true_max
        shared_levels = np.linspace(true_min, true_max, shared_levels)

    # Create plots
    fig, axs = plt.subplots(1, 3, num=fignum, figsize=(9, 3), 
                            sharey='row', constrained_layout=True)#tight_layout=True)

    ax1 = axs[0]
    ax1.set_aspect('equal', adjustable='box')
    ax1.contour(X, Y, Q_vals_1.reshape(len(y), len(x)), levels=5,
                colors='red', alpha=0.5, linewidths=1,
               linestyles='dashed')
    ax1.contour(X, Y, Q_vals_2.reshape(len(y), len(x)), levels=5,
                colors='blue', alpha=0.5, linewidths=1,
               linestyles='dotted')
    cf = ax1.contourf(X, Y, update_dens_1(plotting_params.T).reshape(len(y), len(x)),
                      levels=shared_levels, alpha=1,
                      **plot_dict)
    title_str = ax1_title
    ax1.set_title(title_str)
    ax1.set_xlabel(r'$\lambda_1$')
    ax1.set_ylabel(r'$\lambda_2$')
    
    
    ax2 = axs[1]
    ax2.set_aspect('equal', adjustable='box')
    ax2.contour(X, Y, Q_vals_1.reshape(len(y), len(x)), levels=5,
                colors='red', alpha=0.5, linewidths=1,
               linestyles='dashed')
    ax2.contour(X, Y, Q_vals_2.reshape(len(y), len(x)), levels=5,
                colors='blue', alpha=0.5, linewidths=1,
               linestyles='dotted')
    cf = ax2.contourf(X, Y, update_dens_2(plotting_params.T).reshape(len(y), len(x)),
                      levels=shared_levels,
                      **plot_dict)
    title_str = ax2_title
    ax2.set_title(title_str)
    ax2.set_xlabel(r'$\lambda_1$')
    ax2.set_ylabel(r'$\lambda_2$')

    ax3 = axs[2]
    ax3.set_aspect('equal', adjustable='box')
    cf = ax3.contourf(X, Y, dg_dens(plotting_params.T).reshape(len(y), len(x)),
                      levels=shared_levels,
                      **plot_dict)
    if exact_dg is False:
        title_str = 'Sample DG Distribution'
    else:
        title_str = 'Exact DG Distribution'
    ax3.set_title(title_str)
    ax3.set_xlabel(r'$\lambda_1$')
    ax3.set_ylabel(r'$\lambda_2$')
    
    fig.colorbar(cf, ax=[ax1, ax2, ax3], shrink=0.8)

    if savefig is True:
        plt.savefig(filename, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

### Create function to approximate the KL divergence between two KDEs

In [ ]:
def kl_divergence_kde_grid(kde_p, kde_q, n_grid=100, dim=2, eps=1e-12):
    """
    Approximate KL(P||Q) over the unit box [0,1]^dim using a regular grid.

    Parameters
    ----------
    kde_p, kde_q : scipy.stats.gaussian_kde
        KDE objects.
    n_grid : int
        Number of grid points in each dimension.
    dim : int
        Dimension of the unit box.
    eps : float
        Small positive constant to avoid log(0).

    Returns
    -------
    float
        Approximate KL(P||Q).
    """

    # Construct regular grid
    axes = [np.linspace(0.0, 1.0, n_grid) for _ in range(dim)]
    mesh = np.meshgrid(*axes, indexing="ij")
    points = np.vstack([m.ravel() for m in mesh])  # shape (dim, n_grid**dim)

    # Evaluate KDEs
    p = kde_p(points)
    q = kde_q(points)

    # Renormalize on the unit box
    h = 1.0 / (n_grid - 1)
    cell_volume = h**dim

    p /= np.sum(p) * cell_volume
    q /= np.sum(q) * cell_volume

    # Numerical stability
    p = np.maximum(p, eps)
    q = np.maximum(q, eps)

    return np.sum(p * np.log(p / q)) * cell_volume

## Setup parameter samples

In [ ]:
# Initialize random seed for reproducibility
np.random.seed(121212)

num_init_samples = int(1e4)
param_dim = 2

params_init = np.random.uniform(0, 1, (num_init_samples, param_dim))

In [ ]:
num_dg_samples = int(1e3)

Beta1 = beta(3, 9)
Beta2 = beta(8, 2)
# Make the Beta_dg pdf function evaluate like a KDE
Beta_dg_pdf = lambda x: Beta1.pdf(x[0,:])*Beta2.pdf(x[1,:])

params_dg_1 = Beta1.rvs(size=num_dg_samples)
params_dg_2 = Beta2.rvs(size=num_dg_samples)
params_dg = np.vstack([params_dg_1, params_dg_2]).T

In [ ]:
plt.figure(num=1, figsize=(3, 3))
plt.clf()
plt.scatter(params_init[:, 0], params_init[:, 1], label="Initial")
plt.scatter(params_dg[:, 0], params_dg[:, 1], marker="+", label="DG")
plt.xlabel(r"$\lambda_1$")
plt.ylabel(r"$\lambda_2$")
plt.legend()
plt.tight_layout(pad=0.25)
# plt.savefig('ex1-Beta-params.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

### Choose a skewed or non-skewed map

In [ ]:
# 2D linear maps

skewed = False

if skewed:
    Q_maps = [lambda x: 2*x[:,0] + x[:,1], lambda x: 2.5*x[:,0]+0.5*x[:,1]]  # Copulas have significant impact
else:
    Q_maps = [lambda x: 2*x[:,0] + x[:,1], lambda x: x[:,0]-x[:,1]]  # Copulas have minimal impact

In [ ]:
num_QoI = len(Q_maps)
Q_pred_vals = np.zeros((num_init_samples, num_QoI))
Q_obs_vals = np.zeros((num_dg_samples, num_QoI))

for d in range(num_QoI):
    Q_pred_vals[:, d] = Q_maps[d](params_init)
    Q_obs_vals[:, d] = Q_maps[d](params_dg)

In [ ]:
fig = plt.figure(num=2, figsize=(3,3))
fig.clf()
plt.scatter(Q_pred_vals[:,0], Q_pred_vals[:,1], label='Predicted')
plt.scatter(Q_obs_vals[:,0], Q_obs_vals[:,1], marker='+', label='Observed')
plt.xlabel(r'$Q_1$')
plt.ylabel(r'$Q_2$')
plt.legend()
plt.tight_layout(pad=.25)
# plt.savefig('ex1-Beta-skewed-QoI.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

## Construct iDCI solution

In [ ]:
observed_densities = []
for d in range(num_QoI):
    observed_densities.append(GKDE(Q_obs_vals[:,d]))

In [ ]:
num_epochs = 100

rs_iDCI, last_epoch, last_iter, kl_epochs = iterative_DCI(Q_pred_vals, observed_densities, 
                                           num_epochs = num_epochs, r_init = 1, ratio_tol=0.2,
                                                    KL_tol=1e-7)

In [ ]:
print(np.mean(rs_iDCI[:,-1,:], axis=0))  # looking at diagnostic at end of each epoch

In [ ]:
fignum = 3

interact(plot_epoch_2d_params, 
     fignum = widgets.fixed(fignum),
     epoch_num=IntSlider(min=1, max=last_epoch, steps=1),
     observed_densities=widgets.fixed(observed_densities),
     rs = widgets.fixed(rs_iDCI),
     fix_cbar_ranges=widgets.Checkbox(value=True, 
                                      description='KDE of DG defines all colorbars',
                                      width='auto',
                                      disabled=False),
     cmap=widgets.Text('hot_r'),
     savefig=widgets.Dropdown(options=[True, False], value=False),
     filename=widgets.Text('file.png'))

## Compute approximate copula-correction

In [ ]:
rs_CT_iDCI, _ = copula_transformation(rs_iDCI[:,-1,-1], Q_pred_vals)

In [ ]:
np.mean(rs_CT_iDCI)

In [ ]:
plt.figure(num = 4)
plt.clf()

plt.scatter(rs_iDCI[:,-1,-1], rs_CT_iDCI)
plt.xlabel('iDCI $r$-values')
plt.ylabel('CT-iDCI $r$-values')
# plt.axis('square')
# plt.savefig('ex1-2d-Gaussian-non-skewed-QoI-CT-iDCI-r-values.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

## Compute iDCI solution of CT to re-enforce marginal constraints

In [ ]:
rs_iDCI_CT_iDCI, last_epoch, last_iter, kl_epochs = iterative_DCI(Q_pred_vals, observed_densities,
                                               num_epochs = num_epochs, r_init = rs_CT_iDCI, ratio_tol=0.2,
                                                        KL_tol=1e-7)

In [ ]:
fignum = 5

interact(plot_epoch_2d_params, 
     fignum = widgets.fixed(fignum),
     epoch_num=IntSlider(min=1, max=last_epoch, steps=1),
     observed_densities=widgets.fixed(observed_densities),
     rs = widgets.fixed(rs_iDCI_CT_iDCI),
     fix_cbar_ranges=widgets.Checkbox(value=True, 
                                      description='KDE of DG defines all colorbars',
                                      width='auto',
                                      disabled=False),
     cmap=widgets.Text('hot_r'),
     savefig=widgets.Dropdown(options=[True, False], value=False),
     filename=widgets.Text('file.png'))

## Compare iDCI to CT+iDCI (CT+iDCI is really iDCI+CT+iDCI as mentioned in paper)

In [ ]:
fignum=6

if skewed:
    fname = 'ex1-Betas-skewed-QoI-param-densities.png'
else:
    fname = 'ex1-Betas-nonskewed-QoI-param-densities.png'

plot_2d_params_compare(fignum, rs_iDCI[:,-1,-1], rs_iDCI_CT_iDCI[:,-1,-1], which_params=[0, 1], 
                       cmap='viridis_r',
                       exact_dg = False,
                       savefig = True,
                       filename = fname)

## Evaluate KL divergences

In [ ]:
up_dens_1 = GKDE(params_init.T, weights=rs_iDCI[:, -1, -1])
up_dens_2 = GKDE(params_init.T, weights=rs_iDCI_CT_iDCI[:,-1,-1])

dg_dens = GKDE(params_dg.T)

init_dens = GKDE(params_init.T)
init_dens = lambda x: np.ones(len(x[0,:]))

print('Using KDE of DG Distribution')
print(kl_divergence_kde_grid(dg_dens, init_dens))
print(kl_divergence_kde_grid(dg_dens, up_dens_1))
print(kl_divergence_kde_grid(dg_dens, up_dens_2))

control_tests = 50
dg_control_samples = []
for i in range(control_tests):
    params_dg_1_control = Beta1.rvs(size=num_dg_samples)
    params_dg_2_control = Beta2.rvs(size=num_dg_samples)
    params_dg_control = np.vstack([params_dg_1_control, params_dg_2_control]).T
    dg_control_samples.append(params_dg_control)
    

print('Sample DG Distribution compared to other sample DG Distributions')
# print(kl_divergence_kde_grid(Beta_dg_pdf, dg_dens))
kl_controls = np.zeros(control_tests)
for i in range(control_tests):
    dg_dens_control=GKDE(dg_control_samples[i].T)
    kl_controls[i] = kl_divergence_kde_grid(dg_dens, dg_dens_control)
print(kl_controls.mean(), kl_controls.min(), kl_controls.max())